# 🧠 Neural Networks - Single Layer Perceptron with PyTorch 

<a href="https://colab.research.google.com/github/rubenfonnegra/machine_learning/blob/master/Sem_10/single_layer_perceptron.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> 
<a href="https://github.com/rubenfonnegra/machine_learning/blob/master/Sem_10/single_layer_perceptron.ipynb" target="_parent"><img src="https://img.shields.io/badge/%E2%80%8B-Open%20in%20Github-blue?logo=github" alt="Open In Github"/></a> 

**Two models, one optimization objective: Mean Squared Error (MSE).**

This notebook continues our previous gradient-descent lesson. We start with a single linear neuron (regression unit) and then introduce a neural network with one hidden layer

### Learning objectives

- Translate the familiar equation $\hat y=wx+b$ into PyTorch.
- Understand tensors, trainable parameters, the forward pass, loss, gradients, and weight updates.
- Use PyTorch autograd to calculate gradients while keeping optimization steps explicit.
- Understand why a hidden layer needs a nonlinear activation.
- Train a one-hidden-layer network on nonlinear regression data.
- Compare predictions, MSE, and learning curves.

**Training sequence:** forward pass → MSE → `backward()` → update parameters → clear gradients.


### Documentation

- [```PyTorch```](https://docs.pytorch.org) 
- [```PyTorch Documentation```](https://docs.pytorch.org/docs/2.14/index.html)

---

> **📘 Machine Learning**  
> **Author:** Rubén D. Fonnegra, Ph.D. \
> **Institution:** Institución Universitaria Pascual Bravo  
> © 2026 · Educational use with attribution

## Imports and reproducibility

In [ ]:
#@markdown #### **🛠️⚙️📦 Install complementary dependencies**. 

from tqdm.auto import tqdm
import subprocess, time, sys

LIB = "MLTools-1.2-py3-none-any.whl"
URL = "https://drive.google.com/uc?id=18Y834Tvtj_-Px9yNmbAB4yV4L0gcIZ20"

commands = [
    ("📦 Downloading resources", ["gdown", URL, "-O", LIB], 35),
    ("🔧 Installing dependencies", [sys.executable, "-m", "pip", "install", "-q", LIB], 55),
    ("🧹 Finishing", ["rm", "-f", LIB], 10)
]

print("⚙️ Iniciando configuración del entorno...\n")

try:
    with tqdm(total=100, desc="Preparando", bar_format="{desc}: {bar} {n:.0f}%") as bar:
        for label, command, weight in commands:
            bar.set_description(label)
            subprocess.run(command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            for _ in range(weight):
                time.sleep(0.01)
                bar.update(1)

    print("\n✅ Configuración completada correctamente.")

except subprocess.CalledProcessError as e:
    print("\n❌ Error durante la configuración")
    print(f"Exit code: {e.returncode}")

    if e.stdout:
        print("\n📤 STDOUT:")
        print(e.stdout)

    if e.stderr:
        print("\n🔍 STDERR:")
        print(e.stderr)

    print("\n❌ No fue posible configurar el entorno. Ejecute nuevamente la celda.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from MLTools import generate_linear_regression, generate_nonlinear_regression

import torch
from torch import nn

np.random.seed(42)
torch.manual_seed(42)

## A quick reminder: one neuron

A linear neuron computes

$$\hat y=wx+b$$

Its trainable values are the **weight** $w$ and the **bias** $b$. With regression, we minimize

$$\mathrm{MSE}=\frac{1}{n}\sum_{i=1}^n(y_i-\hat y_i)^2.$$

This is the same mathematical model used in our previous gradient-descent notebook. PyTorch changes *how we represent parameters and compute gradients*, not the underlying objective.

## Part I - Single linear neuron

### Generate a linear dataset

In [ ]:
X, y, y_true = generate_linear_regression(n_samples=240, x_range=(-4, 4), 
                                          slope=2.5, intercept=1.5, 
                                          noise=2.0, random_state=42)

plt.figure(figsize=(6, 4))
plt.scatter(X[:, 0], y, alpha=0.5, label="Observed")
plt.plot(X[:, 0], y_true, label="True relationship")
plt.title("Linear regression dataset")
plt.legend()
plt.show()

### Holdout and PyTorch tensors

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 1)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test.reshape(-1, 1), dtype=torch.float32)

print("X:", X_train.shape, "| y:", y_train.shape)

### Create some helper functions

In [ ]:
def linear_forward(X, w, b):
    return X @ w + b

def mse_loss(y_pred, y_true):
    return torch.mean((y_pred - y_true) ** 2)

### Define the neuron explicitly and apply one training step: 
> forward → backward → update


`torch.nn.Parameter` is a tensor whose gradients PyTorch tracks automatically. \
`loss.backward()` computes the gradients using the chain rule. We then **manually** update the parameters with gradient descent. \
`torch.no_grad()` prevents the update itself from entering the computation graph.

In [ ]:
w = nn.Parameter(torch.tensor([[0.0]]))
b = nn.Parameter(torch.tensor([0.0]))
activation = True
learning_rate = 0.02

prediction = _ 
loss = mse_loss(prediction, y_train)
loss.backward()

print(f"Initial MSE: {loss.item():.4f}")
print(f"dL/dw: {w.grad.item():.4f} | dL/db: {b.grad.item():.4f}")

with torch.no_grad():
    w -= learning_rate * w.grad
    b -= learning_rate * b.grad

w.grad.zero_()
b.grad.zero_()
print(f"After one step: w={w.item():.4f}, b={b.item():.4f}")

### Train the linear neuron

In [ ]:
w = nn.Parameter(torch.tensor([[0.0]]))
b = nn.Parameter(torch.tensor([0.0]))

activation = False
learning_rate = 0.005
epochs = 400
linear_history = []

for epoch in range(epochs):
    prediction = _
    loss = mse_loss(prediction, y_train)
    loss.backward()

    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad

    w.grad.zero_()
    b.grad.zero_()
    linear_history.append(loss.item())

    if epoch % 50 == 0:
        print(f"Epoch {epoch:3d} | MSE={loss.item():.4f} | w={w.item():.3f} | b={b.item():.3f}")

print(f"Learned parameters: y = {w.item():.3f}x + {b.item():.3f}")

### Visualize predictions and convergence

In [ ]:
with torch.no_grad():
    test_pred = linear_forward(X_test, w, b)
    linear_test_mse = mse_loss(test_pred, y_test).item()

x_line = torch.linspace(X_train.min(), X_train.max(), 150).reshape(-1, 1)
with torch.no_grad():
    y_line = linear_forward(x_line, w, b)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(X_train[:, 0].numpy(), y_train[:, 0].numpy(), alpha=0.45, label="Train")
axes[0].scatter(X_test[:, 0].numpy(), y_test[:, 0].numpy(), alpha=0.7, label="Test")
axes[0].plot(x_line.numpy(), y_line.numpy(), linewidth=2, label="Learned neuron")
axes[0].set_title(f"Linear neuron | Test MSE={linear_test_mse:.3f}")
axes[0].legend()
axes[1].plot(linear_history)
axes[1].set(xlabel="Epoch", ylabel="MSE", title="Training loss")
plt.tight_layout()
plt.show()

## Part II - One-hidden-layer neural network

A single linear neuron can only learn a straight line. To learn curved relationships, we combine neurons in a hidden layer and apply a **nonlinear activation**.

For one input and $H$ hidden neurons:

$$\mathbf h=\sigma(x\mathbf W_1+\mathbf b_1)$$

$$\hat y=\mathbf h\mathbf W_2+b_2$$

The output stays **linear** because we are solving a regression problem. The hidden layer uses `Sigmoid` to introduce nonlinearity.

**Architecture:** `1 input → H hidden neurons (Sigmoid) → 1 output`.

### Generate nonlinear regression data

In [ ]:
X_curve, y_curve, y_curve_true = generate_nonlinear_regression( n_samples=300, x_range=(-4, 4), 
                                                                function = "cos",
                                                                noise=0.8, random_state=42)


plt.figure(figsize=(8, 4))
plt.scatter(X_curve[:, 0], y_curve, alpha=0.4, label="Observed")
plt.plot(X_curve[:, 0], y_curve_true, label="Underlying nonlinear function")
plt.title("A curved relationship")
plt.legend()
plt.show()

### Holdout and tensors

In [ ]:
Xh_train, Xh_test, yh_train, yh_test = train_test_split(X_curve, y_curve, test_size = 0.2, random_state = 1)

Xh_train = torch.tensor(Xh_train, dtype=torch.float32)
yh_train = torch.tensor(yh_train.reshape(-1, 1), dtype=torch.float32)
Xh_test = torch.tensor(Xh_test, dtype=torch.float32)
yh_test = torch.tensor(yh_test.reshape(-1, 1), dtype=torch.float32)

### Build the network

Create weights ($W$) and biases ($b$); `nn.Sigmoid` provides nonlinearity. `forward()` defines the operations in the order they are executed.

In [ ]:
class SingleLayerModel:
    def __init__(self, units=16, activation=True):
        self.W1 = (torch.randn(1, units) * 0.1).requires_grad_()
        self.b1 = torch.zeros(units, requires_grad=True)
        
        self.W2 = (torch.randn(units, 1) * 0.1).requires_grad_()
        self.b2 = torch.zeros(1, requires_grad=True)

        self.activation = nn.Sigmoid() if activation else lambda x: x

    def hidden_state(self, X):
        return self.activation(X @ self.W1 + self.b1)
    
    def forward(self, X):
        h = self.hidden_state(X)
        return h @ self.W2 + self.b2

    def parameters(self):
        return [self.W1, self.b1, self.W2, self.b2]


model = SingleLayerModel(units=16)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

### Understand the tensor shapes

For a batch of $n$ examples, the hidden layer produces an $n\times H$ matrix. The output layer combines those hidden activations into $n$ predictions.

In [ ]:
with torch.no_grad():
    sample = Xh_train[:5]
    hidden_values = model.hidden_state(sample)
    output_values = model.forward(sample)

print("Input:", tuple(sample.shape))
print("Hidden activations:", tuple(hidden_values.shape))
print("Output:", tuple(output_values.shape))

### Train with MSE and backpropagation

The optimization logic is unchanged:

1. **Forward pass** produces predictions.
2. **MSE** measures prediction error.
3. **Backpropagation** computes gradients through both layers.
4. **Gradient descent** updates every trainable parameter.

We use `torch.optim.SGD` here to introduce the standard PyTorch training loop. Unlike the previous section, the optimizer performs the parameter updates, but **the learning rate and training loop remain explicit**.

In [ ]:
model = SingleLayerModel(units=16)

criterion = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.02)
epochs = 2500
hidden_history = []

for epoch in range(epochs):
    optimizer.zero_grad()
    predictions = model.forward(Xh_train)
    loss = criterion(predictions, yh_train)
    loss.backward()
    optimizer.step()
    hidden_history.append(loss.item())

    if epoch % 100 == 0:
        print(f"Epoch {epoch:4d} | MSE={loss.item():.4f}")

### Inspect a learned gradient

In [ ]:
optimizer.zero_grad()

predictions = model.forward(Xh_train)
loss = criterion(predictions, yh_train)
loss.backward()
print("First-layer weight gradient shape:", tuple(model.W1.grad.shape))
print("First-layer gradient sample:", model.W1.grad[:5, 0].detach().numpy().round(5))
optimizer.zero_grad()

### Plot the learned nonlinear function and loss

In [ ]:
x_grid = torch.linspace(-4, 4, 400).reshape(-1, 1)

with torch.no_grad():
    curve_pred = model.forward(x_grid)
    hidden_test_mse = criterion(model.forward(Xh_test), yh_test).item()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(Xh_train[:, 0].numpy(), yh_train[:, 0].numpy(), alpha=0.35, label="Train")
axes[0].scatter(Xh_test[:, 0].numpy(), yh_test[:, 0].numpy(), alpha=0.65, label="Test")
axes[0].plot(x_grid.numpy(), curve_pred.numpy(), linewidth=2, label="Neural network")
axes[0].set_title(f"Hidden-layer network | Test MSE={hidden_test_mse:.3f}")
axes[0].legend()
axes[1].plot(hidden_history)
axes[1].set(xlabel="Epoch", ylabel="MSE", title="Training loss")
plt.tight_layout()

### Why do we need activation functions?

If we remove `Sigmoid`, the network becomes

$$\hat y=(x\mathbf W_1+\mathbf b_1) \ \mathbf W_2+b_2$$

which can be rearranged into $\hat y=ax+c$: still a linear function, regardless of the number of hidden neurons.

To demonstrate this, train an identical two-layer architecture *without* the activation function and compare its fit.

In [ ]:
noact_model = SingleLayerModel(units=16, activation=False)
optimizer_linear_stack = torch.optim.SGD(noact_model.parameters(), lr=0.01)
noact_linear_history = []

for epoch in range(2500):
    optimizer_linear_stack.zero_grad()
    loss = criterion(noact_model.forward(Xh_train), yh_train)
    loss.backward()
    optimizer_linear_stack.step()
    noact_linear_history.append(loss.item())

with torch.no_grad():
    stack_pred = noact_model.forward(x_grid)
    stack_mse = criterion(noact_model.forward(Xh_test), yh_test).item()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(Xh_train[:, 0].numpy(), yh_train[:, 0].numpy(), alpha=0.35, label="Train")
axes[0].scatter(Xh_test[:, 0].numpy(), yh_test[:, 0].numpy(), alpha=0.65, label="Test")
axes[0].plot(x_grid.numpy(), stack_pred.numpy(), label=f"No activation (MSE={stack_mse:.3f})")
axes[0].plot(x_grid.numpy(), curve_pred.numpy(), label=f"Tanh hidden layer (MSE={hidden_test_mse:.3f})")
axes[0].set_title(f"Hidden-layer network | Test MSE={hidden_test_mse:.3f}")
axes[0].legend()
axes[1].plot(hidden_history, label=f"Tanh loss curve")
axes[1].plot(noact_linear_history, label=f"No Activation loss curve")
axes[1].set(xlabel="Epoch", ylabel="MSE", title="Training loss")
plt.tight_layout()
plt.show()

## Comparison

| Property | Single linear neuron | One-hidden-layer network |
|---|---|---|
| Architecture | `1 → 1` | `1 → 16 → 1` |
| Nonlinear activation | None | `Sigmoid` |
| Loss | MSE | MSE |
| Optimization | Manual gradient descent | PyTorch SGD |
| Can represent nonlinear functions? | No | Yes |

**Important:** A hidden layer does not automatically guarantee better performance. Its benefit depends on the data, activation function, number of neurons, optimization settings, and generalization.

## Practice exercises

1. Change the number of hidden neurons to `2`, `4`, `8`, and `32`. Compare the curves and test MSE.
2. Change the learning rate to `0.001` and `0.1`. What happens to convergence?
3. Replace `nn.Sigmoid()` with `nn.Tanh()` and `nn.ReLU()`, and compare the learned function.
4. Replace `torch.optim.SGD` with manual parameter updates using `torch.no_grad()` and `parameter.grad`.
5. Try to fit the nonlinear dataset using only a single linear neuron. Explain the result.
6. Explain what `optimizer.zero_grad()`, `loss.backward()`, and `optimizer.step()` each do.

In [ ]:
# Your experiments here


---

## 📄 Attribution

This notebook was developed by **Rubén D. Fonnegra** as educational material for Machine Learning courses at **Institución Universitaria Pascual Bravo**.
You may use, share, and adapt this material for educational purposes, provided that appropriate credit is given to the original author.

**Suggested citation:**
> Fonnegra Tarazona, R. D. (2026). *Neural Networks - Single Layer Perceptron with PyTorch: Machine Learning Notebook*. Institución Universitaria Pascual Bravo.